In [ ]:
# %%capture
# =============================================================================
# UNISIM Validation (single or suite) — Preset-driven, plug-and-play entry point
# =============================================================================
from pathlib import Path
import sys, os, warnings
from typing import List, Dict
import pandas as pd
from IPython.display import display

# Silence TF noise (optional)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("ABSL_LOG_LEVEL", "3")
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow_addons")

# -----------------------------------------------------------------------------
# Bootstrap project root + src import path
# -----------------------------------------------------------------------------
try:
    project_root = Path(get_ipython().run_line_magic("pwd")[0].split("/notebooks")[0])
except Exception:
    project_root = Path.cwd().parent.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from forecast_pipeline.io_utils import configure_logging
configure_logging()

from common.experiment_context import ExperimentContext
from hpo.pivot_validation import (
    run_single_validation_entry,
    run_suite_validation_entry,
    resolve_experiments_for_arch,
)
from hpo.validation_suite import ValidationExperiment

# -----------------------------------------------------------------------------
# Presets (imported from src/forecast_pipeline/arps_offline.py)
# -----------------------------------------------------------------------------
from forecast_pipeline.arps_offline import (
    PIPELINE_PRESETS,
    show_pipeline_presets,
    build_pipeline_config_overrides,
)

# =============================================================================
# Control Panel (ONLY adjust here)
# =============================================================================
arch = "seq2"  # "darts" | "seq2" | "arps"
EXPERIMENT = "HPO_200_Lag_100_Horizon_300"
ENSEMBLE = 1

# ---- User knobs (the only ones you should edit) ----
SEED = 42
PLOT = True
LAG_WINDOW = 100
HORIZON = 300
TEST_SIZE = 0.5
VAL_SIZE = 0.15
RUN_PARAMS = {"ensemble_size": ENSEMBLE, "max_workers": 1}

# ---- Choose ONE preset (default OFF) ----
PIPELINE_PRESET = "OFF"
# Examples:
#   "OFF"
#   "OFFLINE_ANALYTIC_COUPLED_SINGLE"
#   "OFFLINE_ANALYTIC_COUPLED_ENSEMBLE_SPAGHETTI"

# =============================================================================
# Context / run controls
# =============================================================================
CTX = ExperimentContext(group=EXPERIMENT, arch=arch)

MASTER_PROFILE_FILENAME = "final_validation_of_champions.csv"
BASE_RUN_NAME = "validation_seq2"  # results/<family>/validation_seq2
EXECUTION_MODE = "interactive"
CLEAR_RESULTS_BEFORE_RUN = True

RUN_MODE = "single"  # "single" | "suite"
COMPARE_WITH_HPO = True
FORCE_OVERWRITE = True  # If True, auto-confirm deletion of old CSVs in suite mode

# Auto-map architecture name
architecture_name = "Seq2PIN" if arch == "seq2" else None

FILTERS: Dict[str, object] = {
    "dataset": "VOLVE",
    "well": "15/9-F-14",
    "architecture_name": architecture_name,
}

# FILTERS: Dict[str, object] = {
#     "dataset": "UNISIM_IV",
#     # "well": "P11",
#     "architecture_name": architecture_name,
# }


# =============================================================================
# Preset catalog (brief explanation for the user)
# =============================================================================
show_pipeline_presets()

if PIPELINE_PRESET not in PIPELINE_PRESETS:
    raise ValueError(
        f"Unknown PIPELINE_PRESET={PIPELINE_PRESET!r}. "
        f"Choose one of: {sorted(PIPELINE_PRESETS)}"
    )

print(f"\nSelected preset: {PIPELINE_PRESET}")
print(f"Meaning: {PIPELINE_PRESETS[PIPELINE_PRESET].description}\n")

# =============================================================================
# CONFIG_OVERRIDES (built from preset + minimal user knobs)
# =============================================================================
JOB_KNOBS = {
    "seed": SEED,
    "plot": PLOT,
    "lag_window": LAG_WINDOW,
    "horizon": HORIZON,
    "test_size": TEST_SIZE,
    "val_size": VAL_SIZE,
}

CONFIG_OVERRIDES = build_pipeline_config_overrides(
    preset=PIPELINE_PRESET,
    job_knobs=JOB_KNOBS,
    run_params=RUN_PARAMS,
)

# Make sure outputs go to the current experiment folder
CONFIG_OVERRIDES.setdefault("infra", {})["experiments_output_dir"] = str(CTX.results_dir)

# Grid for RUN_MODE="suite" (will be overridden to [reconstruct] if arch="darts")
EXPERIMENTS: List[ValidationExperiment] = [
    ValidationExperiment(policy="reconstruct"),
    ValidationExperiment(policy="hp_hist"),
    ValidationExperiment(policy="reconstruct_warm_raw"),
    ValidationExperiment(policy="reconstruct_warm_hp"),
    ValidationExperiment(policy="reconstruct_warm_ewma"),
    ValidationExperiment(policy="reconstruct_warm_holt"),
]

# =============================================================================
# Main — notebook entry point
# =============================================================================
def main() -> None:
    # Enforce arch-specific experiment policy
    resolved_experiments = resolve_experiments_for_arch(arch, EXPERIMENTS)

    if RUN_MODE == "single":
        _ = run_single_validation_entry(
            project_root=project_root,
            ctx=CTX,
            master_profile_filename=MASTER_PROFILE_FILENAME,
            run_name=BASE_RUN_NAME,
            filters=FILTERS,
            execution_mode=EXECUTION_MODE,
            ensemble_size=ENSEMBLE,
            config_overrides=CONFIG_OVERRIDES,
            delete_previous_results=CLEAR_RESULTS_BEFORE_RUN,
            compare_with_hpo=COMPARE_WITH_HPO,
        )
    elif RUN_MODE == "suite":
        _ = run_suite_validation_entry(
            project_root=project_root,
            ctx=CTX,
            base_run_name=BASE_RUN_NAME,
            master_profile_filename=MASTER_PROFILE_FILENAME,
            filters=FILTERS,
            template_overrides=CONFIG_OVERRIDES,
            experiments=resolved_experiments,
            execution_mode=EXECUTION_MODE,
            ensemble_size=ENSEMBLE,
            force_overwrite=FORCE_OVERWRITE,
        )
    else:
        raise ValueError("RUN_MODE must be either 'single' or 'suite'.")

if __name__ == "__main__":
    main()


In [ ]:
# %%capture
# =============================================================================
# UNISIM Validation (single or suite) — Preset-driven, plug-and-play entry point
# =============================================================================
from pathlib import Path
import sys, os, warnings
from typing import List, Dict
import pandas as pd
from IPython.display import display

# Silence TF noise (optional)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("ABSL_LOG_LEVEL", "3")
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow_addons")

# -----------------------------------------------------------------------------
# Bootstrap project root + src import path
# -----------------------------------------------------------------------------
try:
    project_root = Path(get_ipython().run_line_magic("pwd")[0].split("/notebooks")[0])
except Exception:
    project_root = Path.cwd().parent.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from forecast_pipeline.io_utils import configure_logging
configure_logging()

from common.experiment_context import ExperimentContext
from hpo.pivot_validation import (
    run_single_validation_entry,
    run_suite_validation_entry,
    resolve_experiments_for_arch,
)
from hpo.validation_suite import ValidationExperiment

# -----------------------------------------------------------------------------
# Presets (imported from src/forecast_pipeline/arps_offline.py)
# -----------------------------------------------------------------------------
from forecast_pipeline.arps_offline import (
    PIPELINE_PRESETS,
    show_pipeline_presets,
    build_pipeline_config_overrides,
)

# =============================================================================
# Control Panel (ONLY adjust here)
# =============================================================================
arch = "seq2"  # "darts" | "seq2" | "arps"
EXPERIMENT = "HPO_200_Lag_100_Horizon_300"
ENSEMBLE = 1

# ---- User knobs (the only ones you should edit) ----
SEED = 42
PLOT = True
LAG_WINDOW = 100
HORIZON = 300
TEST_SIZE = 0.55
VAL_SIZE = 0.2
RUN_PARAMS = {"ensemble_size": ENSEMBLE, "max_workers": 1}

# ---- Choose ONE preset (default OFF) ----
PIPELINE_PRESET = "OFF"
# Examples:
#   "OFF"
#   "OFFLINE_ANALYTIC_COUPLED_SINGLE"
#   "OFFLINE_ANALYTIC_COUPLED_ENSEMBLE_SPAGHETTI"

# =============================================================================
# Context / run controls
# =============================================================================
CTX = ExperimentContext(group=EXPERIMENT, arch=arch)

MASTER_PROFILE_FILENAME = "final_validation_of_champions.csv"
BASE_RUN_NAME = "validation_seq2"  # results/<family>/validation_seq2
EXECUTION_MODE = "interactive"
CLEAR_RESULTS_BEFORE_RUN = True

RUN_MODE = "single"  # "single" | "suite"
COMPARE_WITH_HPO = True
FORCE_OVERWRITE = True  # If True, auto-confirm deletion of old CSVs in suite mode

# Auto-map architecture name
architecture_name = "Seq2PIN" if arch == "seq2" else None

FILTERS: Dict[str, object] = {
    "dataset": "VOLVE",
    "well": "15/9-F-14",
    "architecture_name": architecture_name,
}

FILTERS: Dict[str, object] = {
    "dataset": "UNISIM_IV",
    # "well": "P11",
    "architecture_name": architecture_name,
}


# =============================================================================
# Preset catalog (brief explanation for the user)
# =============================================================================
show_pipeline_presets()

if PIPELINE_PRESET not in PIPELINE_PRESETS:
    raise ValueError(
        f"Unknown PIPELINE_PRESET={PIPELINE_PRESET!r}. "
        f"Choose one of: {sorted(PIPELINE_PRESETS)}"
    )

print(f"\nSelected preset: {PIPELINE_PRESET}")
print(f"Meaning: {PIPELINE_PRESETS[PIPELINE_PRESET].description}\n")

# =============================================================================
# CONFIG_OVERRIDES (built from preset + minimal user knobs)
# =============================================================================
JOB_KNOBS = {
    "seed": SEED,
    "plot": PLOT,
    "lag_window": LAG_WINDOW,
    "horizon": HORIZON,
    "test_size": TEST_SIZE,
    "val_size": VAL_SIZE,
}

CONFIG_OVERRIDES = build_pipeline_config_overrides(
    preset=PIPELINE_PRESET,
    job_knobs=JOB_KNOBS,
    run_params=RUN_PARAMS,
)

# Make sure outputs go to the current experiment folder
CONFIG_OVERRIDES.setdefault("infra", {})["experiments_output_dir"] = str(CTX.results_dir)

# Grid for RUN_MODE="suite" (will be overridden to [reconstruct] if arch="darts")
EXPERIMENTS: List[ValidationExperiment] = [
    ValidationExperiment(policy="reconstruct"),
    ValidationExperiment(policy="hp_hist"),
    ValidationExperiment(policy="reconstruct_warm_raw"),
    ValidationExperiment(policy="reconstruct_warm_hp"),
    ValidationExperiment(policy="reconstruct_warm_ewma"),
    ValidationExperiment(policy="reconstruct_warm_holt"),
]

# =============================================================================
# Main — notebook entry point
# =============================================================================
def main() -> None:
    # Enforce arch-specific experiment policy
    resolved_experiments = resolve_experiments_for_arch(arch, EXPERIMENTS)

    if RUN_MODE == "single":
        _ = run_single_validation_entry(
            project_root=project_root,
            ctx=CTX,
            master_profile_filename=MASTER_PROFILE_FILENAME,
            run_name=BASE_RUN_NAME,
            filters=FILTERS,
            execution_mode=EXECUTION_MODE,
            ensemble_size=ENSEMBLE,
            config_overrides=CONFIG_OVERRIDES,
            delete_previous_results=CLEAR_RESULTS_BEFORE_RUN,
            compare_with_hpo=COMPARE_WITH_HPO,
        )
    elif RUN_MODE == "suite":
        _ = run_suite_validation_entry(
            project_root=project_root,
            ctx=CTX,
            base_run_name=BASE_RUN_NAME,
            master_profile_filename=MASTER_PROFILE_FILENAME,
            filters=FILTERS,
            template_overrides=CONFIG_OVERRIDES,
            experiments=resolved_experiments,
            execution_mode=EXECUTION_MODE,
            ensemble_size=ENSEMBLE,
            force_overwrite=FORCE_OVERWRITE,
        )
    else:
        raise ValueError("RUN_MODE must be either 'single' or 'suite'.")

if __name__ == "__main__":
    main()


In [ ]:
"""
Master Dashboard for HPO Champion Selection (Notebook Entry Point)

This cell is the ONLY place you should edit to switch selection behavior.

Four explicit modes (no hybrids):
  1) LEGACY_WEIGHTED    -> legacy gates, selection_col='weighted_score'
  2) LEGACY_ROBUSTCOL   -> legacy gates, selection_col='robust_score'
  3) NEIGHBOR_TOP_PCT   -> neighborhood robust selector, pool_method='top_pct' (VAL-only), TEST audit-only
  4) NEIGHBOR_VAL_BAND  -> neighborhood robust selector, pool_method='val_band' (VAL-only), TEST audit-only

Key semantics:
  - selector_mode controls WHICH selection path is executed (LEGACY vs NEIGHBOR).
  - scoring_strategy controls HOW scores are PRODUCED during aggregation (e.g., weighted_score / robust_score).
    These are intentionally independent.
"""
from __future__ import annotations

import sys
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Mapping

import pandas as pd


# ==============================================================================
# Project setup
# ==============================================================================
try:
    project_root = Path(__file__).parent.parent.parent  # script
except NameError:
    project_root = Path.cwd().parent.parent  # notebook

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from forecast_pipeline.io_utils import configure_logging
from forecast_pipeline.plotting import render_champions_view_auto
from common.experiment_context import ExperimentContext

from hpo.analysis import SelectionRunConfig, run_selection_pipeline

configure_logging()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

try:
    from IPython.display import display
except ImportError:
    display = print


# ==============================================================================
# Presets (edit only this section)
# ==============================================================================

@dataclass(frozen=True)
class ModePreset:
    name: str
    selector_mode: str
    scoring_strategy: str
    metric_to_optimize: str
    # Optional: only used by NEIGHBOR_* path (wired in run_selection_pipeline)
    neighborhood_overrides: Dict[str, Any]
    # Optional: legacy posthoc overrides (still used in LEGACY_* path)
    posthoc_overrides: Dict[str, Any]
    help: str


# --- Global toggles ---
GROUP = "HPO_200_Lag_100_Horizon_300"
ARCH_SELECTION = "all"  # "all" | "seq2" | "darts" | "arps"

PLOT_SUMMARY_BARS = True
PLOT_ARCH_PERF = True
PLOT_HPARAM_IMPORTANCE = True
PLOT_CHAMPIONS = True

VALIDATION_RUN_NAME = "final_validation_of_champions"
VALIDATION_SEED = 42

SEQ2_ARCH_FILTER = "Seq2PIN"  # or None


# --- Scoring inputs used by aggregation ---
METRIC_WEIGHTS = {
    "val_smape_cum": 1.0,
    "val_smape_agg": 5.0,
}

# IMPORTANT:
# Use YOUR project's source-of-truth ordering resolver in code, but still provide this mapping
# so older parts of the stack don't flip silently.
LOWER_IS_BETTER = {
    "val_smape_cum": True,
    "val_smape_agg": True,
    "weighted_score": True,
    "robust_score": True,
}


# --- Legacy posthoc overrides (used only by LEGACY_* modes) ---
POSTHOC_OVERRIDES_COMMON = dict(
    top_strategies_per_well=2,
    per_strategy_k=10,
    selection_strategy="best_of_the_best",
    apply_pareto=False,
    primary_quantile={"val_smape_agg": 0.6},
    mad_guard={"enabled": False, "alpha": 1, "metrics": ["val_smape_cum", "val_smape_agg"], "log": True, "side": "right"},
    relax_pool=False,
    valcum_gate={"q_low": 0.05, "q_high": 0.8},
)

POSTHOC_OVERRIDES_ARPS = {**POSTHOC_OVERRIDES_COMMON, "hpo_signature_cols": [
    "well", "campaign", "variant", "solver", "weighting", "loss", "lag_window", "horizon",
]}

POSTHOC_OVERRIDES_SEQ2 = {**POSTHOC_OVERRIDES_COMMON, "hpo_signature_cols": [
    "well", "campaign", "physics_strategy", "epochs", "batch_size", "learning_rate", "data_sample",
]}

POSTHOC_OVERRIDES_DARTS = {**POSTHOC_OVERRIDES_COMMON, "hpo_signature_cols": [
    "well", "campaign", "profile", "physics_strategy", "n_epochs", "batch_size", "learning_rate",
]}


# --- Neighborhood overrides (used only by NEIGHBOR_* modes) ---
DEFAULT_NEIGHBORHOOD = dict(
    # pick_pool_idx config
    pool_cfg=dict(top_pct=0.1, drop=0.05, take=0.40, min_candidates=20),
    # robust scoring config
    robust_cfg=dict(k=20, min_strat=25, alpha=0.65, beta=0.03, gamma=0.35, luck_q=0.25, w_lr=2.0, w_ep=0.25, w_bs=0.50),
    # grouping keys (optional)
    group_cols=["dataset", "well", "architecture"],
)

# --- Explicit modes ---
PRESETS: Dict[str, ModePreset] = {
    "LEGACY_WEIGHTED": ModePreset(
        name="LEGACY_WEIGHTED",
        selector_mode="LEGACY_WEIGHTED",
        scoring_strategy="weighted_score",      # aggregation produces weighted_score
        metric_to_optimize="weighted_score",    # selection orders by weighted_score
        neighborhood_overrides={},
        posthoc_overrides={},
        help="Legacy gates + dedup + champions chosen by weighted_score (lowe is better). Pool is locked to survivors.",
    ),
    "LEGACY_ROBUSTCOL": ModePreset(
        name="LEGACY_ROBUSTCOL",
        selector_mode="LEGACY_ROBUSTCOL",
        scoring_strategy="robust_score",        # aggregation produces robust_score
        metric_to_optimize="robust_score",      # selection orders by robust_score
        neighborhood_overrides={},
        posthoc_overrides={},
        help="Legacy gates + dedup + champions chosen by robust_score (lower is better). Pool is locked to survivors.",
    ),
    "NEIGHBOR_TOP_PCT": ModePreset(
        name="NEIGHBOR_TOP_PCT",
        selector_mode="NEIGHBOR_TOP_PCT",
        scoring_strategy="robust_score",        # you can keep producing robust_score, but selection is neighborhood-based
        metric_to_optimize="robust_score",      # contract/meta label; neighborhood writes robust_score column too
        neighborhood_overrides={**DEFAULT_NEIGHBORHOOD, "pool_method": "top_pct"},
        posthoc_overrides={},
        help="Neighborhood robust selector: pool = top_pct (VAL-only), kNN local scoring in HP-space, TEST is audit-only.",
    ),
    "NEIGHBOR_VAL_BAND": ModePreset(
        name="NEIGHBOR_VAL_BAND",
        selector_mode="NEIGHBOR_VAL_BAND",
        scoring_strategy="robust_score",
        metric_to_optimize="robust_score",
        neighborhood_overrides={**DEFAULT_NEIGHBORHOOD, "pool_method": "val_band"},
        posthoc_overrides={},
        help="Neighborhood robust selector: pool = val_band (VAL-only), kNN local scoring in HP-space, TEST is audit-only.",
    ),
}

# >>> Choose the mode here <<<
MODE = "LEGACY_ROBUSTCOL"  # <- change me


# ==============================================================================
# Runner (no edits typically needed below)
# ==============================================================================

def _arch_list(selection: str) -> List[str]:
    s = (selection or "all").lower().strip()
    return [s] if s in {"seq2", "darts", "arps"} else ["seq2", "darts", "arps"]

def _arch_filter(arch: str) -> Optional[str]:
    return SEQ2_ARCH_FILTER if arch == "seq2" else None

def _arch_overrides(arch: str) -> Dict[str, Any]:
    return {"seq2": POSTHOC_OVERRIDES_SEQ2, "arps": POSTHOC_OVERRIDES_ARPS, "darts": POSTHOC_OVERRIDES_DARTS}.get(
        arch, POSTHOC_OVERRIDES_COMMON
    )

def _resolve_champions_columns(df: pd.DataFrame, primary_metric: str, score_col: str) -> List[str]:
    if df is None or df.empty:
        return []
    def _has(c: str) -> bool: return c in df.columns

    family_cols = []
    if _has("variant") and _has("solver"):
        family_cols = ["variant", "solver", "weighting", "loss"]
    elif _has("profile") and _has("n_epochs"):
        family_cols = ["architecture_name", "physics_strategy", "profile", "n_epochs", "batch_size", "learning_rate"]
    elif _has("physics_strategy") and _has("epochs"):
        family_cols = ["architecture_name", "physics_strategy", "aggregation_method", "data_sample", "epochs", "learning_rate", "batch_size"]

    base = ["rank", "well", primary_metric, score_col]
    if score_col == "robust_score":
        base += ["_neighbor_scope", "_neighbor_local_mad", "_neighbor_rank_gap"]  # neighborhood diagnostics if present
    cols = list(dict.fromkeys([c for c in (base + family_cols) if c in df.columns]))
    return cols

def _print_mode_help(p: ModePreset) -> None:
    print("\n" + "=" * 90)
    print(f"MODE: {p.name}")
    print(p.help)
    print(f"selector_mode     : {p.selector_mode}")
    print(f"scoring_strategy  : {p.scoring_strategy}")
    print(f"metric_to_optimize: {p.metric_to_optimize}")
    if p.selector_mode.startswith("NEIGHBOR_"):
        nm = p.neighborhood_overrides.get("pool_method", "unknown")
        cfg = p.neighborhood_overrides.get("pool_cfg", {})
        print(f"neighborhood pool : {nm} | pool_cfg={cfg}")
    print("=" * 90 + "\n")

def run_one_architecture(arch: str, preset: ModePreset) -> Dict[str, Any]:
    ctx = ExperimentContext(group=GROUP, arch=arch)

    # Compose legacy posthoc overrides (only used by legacy path in run_selection_pipeline)
    posthoc_over = dict(_arch_overrides(arch))
    posthoc_over.update(preset.posthoc_overrides or {})

    cfg = SelectionRunConfig(
        results_dir=ctx.results_dir,
        reports_dir=ctx.reports_dir_arch,
        metric_weights=METRIC_WEIGHTS,
        lower_is_better=LOWER_IS_BETTER,
        scoring_strategy=preset.scoring_strategy,
        metric_to_optimize=preset.metric_to_optimize,
        selector_mode=preset.selector_mode,
        neighborhood_overrides=dict(preset.neighborhood_overrides or {}),
        posthoc_overrides=posthoc_over,
        arch_filter=_arch_filter(arch),
        plot_architecture_performance=PLOT_ARCH_PERF,
        plot_hparam_importance_per_well=PLOT_HPARAM_IMPORTANCE,
        plot_champions_per_well=PLOT_CHAMPIONS,
        validation_run_name=VALIDATION_RUN_NAME,
        validation_seed=VALIDATION_SEED,
        plot_summary_bars=PLOT_SUMMARY_BARS,
    )

    return run_selection_pipeline(cfg)

def render_result(result: Mapping[str, Any], preset: ModePreset) -> None:
    top = result.get("top_performers")
    meta = result.get("meta", {}) or {}
    summary = result.get("summary")

    print("\n--- OUTPUT -----------------------------")
    print(f"leaderboard_path      : {result.get('leaderboard_path')}")
    print(f"validation_profile    : {result.get('validation_profile_path')}")
    print(f"top_performers rows   : {len(top) if isinstance(top, pd.DataFrame) else 0}")
    print(f"summary rows          : {len(summary) if isinstance(summary, pd.DataFrame) else 0}")
    print(f"meta.selection_path   : {meta.get('selection_path')}")
    print(f"meta.selector_mode    : {meta.get('selector_mode')}")
    print(f"meta.selection_col    : {meta.get('selection_col')}")
    print(f"meta.pool_method      : {meta.get('pool_method')}")
    print(f"meta.score_direction  : {meta.get('score_direction')}")

    # --------------------------
    # Champions view
    # --------------------------
    if isinstance(top, pd.DataFrame) and not top.empty:
        primary_metric = "val_smape_agg"
        score_col = str(meta.get("selection_col") or preset.metric_to_optimize)
        cols = _resolve_champions_columns(top, primary_metric, score_col)

        print("\n--- 🏆 Champions View 🏆 ---")
        styled = render_champions_view_auto(
            df=top[cols] if cols else top,
            per_well_k=2,
            metric=primary_metric,
            lower_is_better=True,
        )
        display(styled)
    else:
        print("\nNo champions were selected (or mode not implemented).")

    # --------------------------
    # Regret table (compact)
    # --------------------------
    if isinstance(summary, pd.DataFrame) and not summary.empty:
        print("\n--- 📉 Regret (TEST audit-only) ---")

        # 1. Select Columns (Snake case logic)
        wanted = [
            "dataset", "well", "architecture",
            "chosen_val_smape_agg",
            "chosen_test_smape_agg",
            "pool_best_test_smape_agg",
            "regret_test",
            "ratio_test",
            "val_test_spearman",
            "chosen_test_percentile",
        ]
        cols = [c for c in wanted if c in summary.columns]
        out = summary[cols].copy()

        # 2. Process Values (Numeric conversion + Rounding)
        # We do this BEFORE renaming to keep logic simple
        num_cols = [c for c in out.columns if c not in {"dataset", "well", "architecture"}]
        for c in num_cols:
            out[c] = pd.to_numeric(out[c], errors="coerce")

        out = out.round(2)

        # 3. Sort (Stable sort on keys)
        sort_cols = [c for c in ["dataset", "well", "architecture"] if c in out.columns]
        if sort_cols:
            out = out.sort_values(sort_cols, kind="mergesort")
        
        # 4. Rename for Display (Compact Names)
        column_mapping = {
            "dataset": "Dataset",
            "well": "Well",
            "architecture": "Arch",
            "chosen_val_smape_agg": "Chosen Val",
            "chosen_test_smape_agg": "Chosen Test",
            "pool_best_test_smape_agg": "Best Test",
            "regret_test": "Regret Test",
            "ratio_test": "Ratio Test",
            "val_test_spearman": "Spearman",
            "chosen_test_percentile": "Test Pctl"
        }
        
        display(out.rename(columns=column_mapping))

        # Helpful hint if Spearman isn't present yet
        if "val_test_spearman" not in summary.columns:
            print("NOTE: 'val_test_spearman' is not in summary yet. Add it in build_canonical_summary.")

        summary = result.get("summary")
        render_quick_audit(summary)


def render_quick_audit(summary):
    import numpy as np
    import pandas as pd

    if summary is None or not isinstance(summary, pd.DataFrame) or summary.empty:
        return

    def q(x: np.ndarray, p: float) -> float:
        x = x[np.isfinite(x)]
        return float(np.quantile(x, p)) if len(x) else np.nan

    regret = pd.to_numeric(summary.get("regret_test", np.nan), errors="coerce").to_numpy(dtype=float)
    ratio = pd.to_numeric(summary.get("ratio_test", np.nan), errors="coerce").to_numpy(dtype=float)
    spearman = pd.to_numeric(summary.get("val_test_spearman", np.nan), errors="coerce").to_numpy(dtype=float)

    print("\n=== Quick audit (campaign-wide; TEST is audit-only) ===")
    print("Spearman(VAL,TEST) is a rank correlation (range [-1, +1]). +1 means VAL ranking matches TEST ranking (good proxy); 0 means weak/no relationship; -1 means VAL ranking is inverted vs TEST (risky proxy).")
    print("TEST regret = chosen_TEST - best_TEST (absolute gap; 0 is perfect; lower is better). TEST ratio = chosen_TEST / best_TEST (relative gap; 1.0 is perfect; e.g., 1.21 means the chosen model is ~21% worse than the best TEST for that group).")
    print("Quantiles: median (p50) is the typical case; p90 means 90% of groups are at or below that value (so it's a 'near-worst-case' summary).\n")

    print(f"- TEST regret: median={q(regret, 0.5):.4g} | p90={q(regret, 0.9):.4g} | max={np.nanmax(regret) if np.isfinite(regret).any() else np.nan:.6g}")
    print(f"- TEST ratio : median={q(ratio, 0.5):.4g} | p90={q(ratio, 0.9):.4g} | max={np.nanmax(ratio) if np.isfinite(ratio).any() else np.nan:.6g}")
    print(f"- Spearman(VAL,TEST): median={q(spearman, 0.5):.4g} | min={np.nanmin(spearman) if np.isfinite(spearman).any() else np.nan:.6g} | max={np.nanmax(spearman) if np.isfinite(spearman).any() else np.nan:.6g}")

    n_bad = int(np.sum(np.isfinite(spearman) & (spearman < 0)))
    if n_bad:
        print(f"\n⚠️  Note: {n_bad}/{len(summary)} groups have negative Spearman. In those groups, VAL ranking may be a poor proxy for TEST ranking.")



# ==============================================================================
# Execute
# ==============================================================================
preset = PRESETS.get(str(MODE).upper().strip())
if preset is None:
    raise ValueError(f"Unknown MODE='{MODE}'. Available={list(PRESETS.keys())}")

_print_mode_help(preset)

for arch in _arch_list(ARCH_SELECTION):
    print("\n" + "-" * 90)
    print(f"Running ARCH={arch.upper()} | GROUP={GROUP} | MODE={preset.name}")
    print("-" * 90)
    try:
        res = run_one_architecture(arch, preset)
        render_result(res, preset)
    except Exception as e:
        logging.error("❌ %s RUN FAILED: %s: %s", arch.upper(), type(e).__name__, e, exc_info=True)
